In [0]:
import sys
sys.path.append('/Workspace/Repos/MHS-analytics/MHS-analytics/Python_Packages')

from MHS_Helper.mhs_db_variables import get_mhs_back_end_auto_connection, get_mhs_ops_mart_auto_connection
from DB.db_modes import JDBC_MODES

mhs_backend_conn = get_mhs_back_end_auto_connection(db_type='production')

query = """
SELECT TOP 10
       mv.[reportingDate]
      ,mv.[value]
FROM [dbo].[MeasureValue] mv
WHERE mv.[reportingDate] > '2023-03-31'
"""

back_end_df = mhs_backend_conn.read_from_db(query)
display(back_end_df)

In [0]:
import sys
sys.path.append('/Workspace/Repos/MHS-analytics/MHS-analytics/Python_Packages')

from MHS_Helper.mhs_db_variables import get_mhs_back_end_auto_connection
from DB.db_modes import JDBC_MODES

mhs_backend_conn = get_mhs_back_end_auto_connection(db_type='production')

query = """
SELECT
       mv.[reportingDate]
      ,mv.[value]
      ,m.[InternalID]
FROM [dbo].[MeasureValue] mv
JOIN [dbo].[Measure] m
    ON [MeasureID] = m.ID
WHERE mv.[reportingDate] > '2023-03-31'
  AND m.[InternalID] LIKE 'OZ%'
"""

back_end_df = mhs_backend_conn.read_from_db(query)

dfs = {}
dfs["hist"] = back_end_df

display(dfs["hist"])

row_count = dfs["hist"].count()
display(spark.createDataFrame([(row_count,)], ["row_count"]))

In [0]:
import sys
sys.path.append('/Workspace/Repos/MHS-analytics/MHS-analytics/Python_Packages')

from MHS_Helper.mhs_db_variables import get_mhs_back_end_auto_connection
from DB.db_modes import JDBC_MODES
import openpyxl

# 1. Connect
mhs_backend_conn = get_mhs_back_end_auto_connection(db_type='production')

# 2. Read the OZ lists from Excel
xlsx_path = "/Workspace/Users/steven.evans4@udal.nhs.uk/MHS-ingestion-for-Outpatients/op_ref.xlsx"
sheet_name = "Lists and df"

groups = [
    ("df_fofu", 4),
    ("df_miss_all", 9),
    ("df_miss_fst", 14),
    ("df_miss_f2f", 19),
    ("df_miss_2ww", 24),
    ("df_fu_nop", 29),
    ("df_remote", 34),
    ("df_all", 39),
]

def extract_metric_id_lists(xlsx_path: str, sheet_name: str, groups, start_row: int = 6):
    wb = openpyxl.load_workbook(xlsx_path, data_only=True)
    if sheet_name not in wb.sheetnames:
        raise ValueError(f"Sheet '{sheet_name}' not found. Available sheets: {wb.sheetnames}")

    ws = wb[sheet_name]
    out = {}

    for name, metric_col in groups:
        ids = []
        for r in range(start_row, ws.max_row + 1):
            v = ws.cell(r, metric_col).value

            if v is None or str(v).strip() == "":
                if ids:
                    lookahead_end = True
                    for rr in range(r, min(r + 6, ws.max_row + 1)):
                        vv = ws.cell(rr, metric_col).value
                        if vv is not None and str(vv).strip() != "":
                            lookahead_end = False
                            break
                    if lookahead_end:
                        break
                continue

            ids.append(str(v).strip())

        seen = set()
        ids = [x for x in ids if not (x in seen or seen.add(x))]

        if len(ids) == 0:
            print(f"WARNING: '{name}' list extracted 0 Metric IDs.")

        out[name] = ids

    return out

metric_lists = extract_metric_id_lists(xlsx_path, sheet_name, groups)

# 3. Combine all extracted codes into one unique list
all_metric_ids = []
seen = set()

for ids in metric_lists.values():
    for x in ids:
        if x.startswith("OZ") and x not in seen:
            seen.add(x)
            all_metric_ids.append(x)

print("Number of OZ codes:", len(all_metric_ids))
print("Example OZ codes:", all_metric_ids[:10])

# 4. Build SQL IN clause
if not all_metric_ids:
    raise ValueError("No OZ metric IDs found in the Excel lists.")

metric_id_sql = ", ".join(f"'{x}'" for x in all_metric_ids)

query = f"""
SELECT
       mv.[reportingDate]
      ,mv.[value]
      ,m.[InternalID]
FROM [dbo].[MeasureValue] mv
JOIN [dbo].[Measure] m
    ON [MeasureID] = m.ID
WHERE mv.[reportingDate] > '2023-03-31'
  AND m.[InternalID] IN ({metric_id_sql})
"""

back_end_df = mhs_backend_conn.read_from_db(query)

dfs = {}
dfs["hist"] = back_end_df

display(dfs["hist"])

row_count = dfs["hist"].count()
display(spark.createDataFrame([(row_count,)], ["row_count"]))

In [0]:
# 1 Import the (OZ) base lists from op_ref.xlsx
import openpyxl
import sys

sys.path.append('/Workspace/Repos/MHS-analytics/MHS-analytics/Python_Packages')

from MHS_Helper.mhs_db_variables import get_mhs_back_end_auto_connection
from DB.db_modes import JDBC_MODES


# 2 Excel source
xlsx_path = "/Workspace/Users/steven.evans4@udal.nhs.uk/MHS-ingestion-for-Outpatients/op_ref.xlsx"
sheet_name = "Lists and df"

groups = [
    ("df_fofu", 4),
    ("df_miss_all", 9),
    ("df_miss_fst", 14),
    ("df_miss_f2f", 19),
    ("df_miss_2ww", 24),
    ("df_fu_nop", 29),
    ("df_remote", 34),
    ("df_all", 39),
]


# 3 Function to extract OZ metric lists
def extract_metric_id_lists(xlsx_path: str, sheet_name: str, groups, start_row: int = 6):

    wb = openpyxl.load_workbook(xlsx_path, data_only=True)

    if sheet_name not in wb.sheetnames:
        raise ValueError(f"Sheet '{sheet_name}' not found. Available sheets: {wb.sheetnames}")

    ws = wb[sheet_name]

    out = {}

    for name, metric_col in groups:

        ids = []

        for r in range(start_row, ws.max_row + 1):

            v = ws.cell(r, metric_col).value

            if v is None or str(v).strip() == "":

                if ids:

                    lookahead_end = True

                    for rr in range(r, min(r + 6, ws.max_row + 1)):

                        vv = ws.cell(rr, metric_col).value

                        if vv is not None and str(vv).strip() != "":
                            lookahead_end = False
                            break

                    if lookahead_end:
                        break

                continue

            ids.append(str(v).strip())

        # remove duplicates but preserve order
        seen = set()
        ids = [x for x in ids if not (x in seen or seen.add(x))]

        if len(ids) == 0:
            print(f"WARNING: '{name}' list extracted 0 Metric IDs.")

        out[name] = ids

    return out


metric_lists = extract_metric_id_lists(xlsx_path, sheet_name, groups)


# 4 Combine all OZ codes from the lists
all_metric_ids = []
seen = set()

for ids in metric_lists.values():
    for x in ids:
        if x.startswith("OZ") and x not in seen:
            seen.add(x)
            all_metric_ids.append(x)

print("Number of OZ codes:", len(all_metric_ids))
print("Example codes:", all_metric_ids[:10])


# 5 Build SQL IN clause
metric_id_sql = ", ".join(f"'{x}'" for x in all_metric_ids)


# 6 Connect to SQL Server
mhs_backend_conn = get_mhs_back_end_auto_connection(db_type='production')


# 7 Query with cleaned reportingDate
query = f"""
SELECT
       CAST(mv.[reportingDate] AS DATE) AS reportingDate
      ,mv.[value]
      ,m.[InternalID]
FROM [dbo].[MeasureValue] mv
JOIN [dbo].[Measure] m
    ON [MeasureID] = m.ID
WHERE mv.[reportingDate] > '2023-03-31'
  AND m.[InternalID] IN ({metric_id_sql})
"""


# 8 Run query
back_end_df = mhs_backend_conn.read_from_db(query)


# 9 Store dataframe
dfs = {}
dfs["hist"] = back_end_df


# 10 Display results
display(dfs["hist"])


# 11 Row count
row_count = dfs["hist"].count()
display(spark.createDataFrame([(row_count,)], ["row_count"]))

In [0]:
# 1 Import the (OZ) base lists from op_ref.xlsx
import openpyxl
import sys

sys.path.append('/Workspace/Repos/MHS-analytics/MHS-analytics/Python_Packages')

from MHS_Helper.mhs_db_variables import get_mhs_back_end_auto_connection
from DB.db_modes import JDBC_MODES


# 2 Excel source
xlsx_path = "/Workspace/Users/steven.evans4@udal.nhs.uk/MHS-ingestion-for-Outpatients/op_ref.xlsx"
sheet_name = "Lists and df"

groups = [
    ("df_fofu", 4),
    ("df_miss_all", 9),
    ("df_miss_fst", 14),
    ("df_miss_f2f", 19),
    ("df_miss_2ww", 24),
    ("df_fu_nop", 29),
    ("df_remote", 34),
    ("df_all", 39),
]


# 3 Function to extract OZ metric lists
def extract_metric_id_lists(xlsx_path: str, sheet_name: str, groups, start_row: int = 6):

    wb = openpyxl.load_workbook(xlsx_path, data_only=True)

    if sheet_name not in wb.sheetnames:
        raise ValueError(f"Sheet '{sheet_name}' not found. Available sheets: {wb.sheetnames}")

    ws = wb[sheet_name]
    out = {}

    for name, metric_col in groups:
        ids = []

        for r in range(start_row, ws.max_row + 1):
            v = ws.cell(r, metric_col).value

            if v is None or str(v).strip() == "":
                if ids:
                    lookahead_end = True
                    for rr in range(r, min(r + 6, ws.max_row + 1)):
                        vv = ws.cell(rr, metric_col).value
                        if vv is not None and str(vv).strip() != "":
                            lookahead_end = False
                            break
                    if lookahead_end:
                        break
                continue

            ids.append(str(v).strip())

        # remove duplicates but preserve order
        seen = set()
        ids = [x for x in ids if not (x in seen or seen.add(x))]

        if len(ids) == 0:
            print(f"WARNING: '{name}' list extracted 0 Metric IDs.")

        out[name] = ids

    return out


metric_lists = extract_metric_id_lists(xlsx_path, sheet_name, groups)


# 4 Combine all OZ codes from the lists
all_metric_ids = []
seen = set()

for ids in metric_lists.values():
    for x in ids:
        if x.startswith("OZ") and x not in seen:
            seen.add(x)
            all_metric_ids.append(x)

print("Number of OZ codes:", len(all_metric_ids))
print("Example codes:", all_metric_ids[:10])


# 5 Build SQL IN clause
metric_id_sql = ", ".join("'" + x.replace("'", "''") + "'" for x in all_metric_ids)


# 6 Connect to SQL Server
mhs_backend_conn = get_mhs_back_end_auto_connection(db_type='production')


# 7 Query with cleaned reportingDate + providerCode
query = f"""
SELECT
       CAST(mv.[reportingDate] AS DATE) AS reportingDate
      ,CAST(mv.[value] AS FLOAT) AS value
      ,m.[InternalID] AS metricID
      ,p.[Code] AS providerCode
FROM [dbo].[MeasureValue] mv
JOIN [dbo].[Measure] m
    ON [MeasureID] = m.ID
LEFT JOIN [dbo].[Provider] p
    ON p.ID = mv.ProviderID
WHERE mv.[reportingDate] > '2023-03-31'
  AND m.[InternalID] IN ({metric_id_sql})
"""


# 8 Run query
prod_df = mhs_backend_conn.read_from_db(query)


# 9 Store dataframe
dfs = {}
dfs["hist"] = prod_df


# 10 Display results
display(dfs["hist"])


# 11 Row count
row_count = dfs["hist"].count()
display(spark.createDataFrame([(row_count,)], ["row_count"]))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, StringType, DoubleType

# ----------------------------
# A. Standardise Prod
# ----------------------------
prod_compare = (
    prod_df
    .withColumn("reportingDate", F.col("reportingDate").cast(DateType()))
    .withColumn("metricID", F.trim(F.col("metricID")).cast(StringType()))
    .withColumn("providerCode", F.trim(F.col("providerCode")).cast(StringType()))
    .withColumn("value", F.col("value").cast(DoubleType()))
    .select("reportingDate", "metricID", "providerCode", "value")
)

# ----------------------------
# B. Standardise Preview
# ----------------------------
preview_compare = (
    df_outpat_metrics
    .withColumn("reportingDate", F.col("reportingDate").cast(DateType()))
    .withColumn("metricID", F.trim(F.col("metricID")).cast(StringType()))
    .withColumn("providerCode", F.trim(F.col("providerCode")).cast(StringType()))
    .withColumn("value", F.regexp_replace(F.col("value"), ",", "").cast(DoubleType()))
    .select("reportingDate", "metricID", "providerCode", "value")
)

# ----------------------------
# C. Aggregate each side to the same grain
# ----------------------------
prod_agg = (
    prod_compare
    .groupBy("reportingDate", "metricID", "providerCode")
    .agg(F.sum("value").alias("prod_value"))
)

preview_agg = (
    preview_compare
    .groupBy("reportingDate", "metricID", "providerCode")
    .agg(F.sum("value").alias("preview_value"))
)

# ----------------------------
# D. Compare Preview vs Prod
# difference_value = preview_value - prod_value
# ----------------------------
comparison_df = (
    preview_agg.alias("prv")
    .join(
        prod_agg.alias("prd"),
        on=["reportingDate", "metricID", "providerCode"],
        how="full_outer"
    )
    .select(
        F.coalesce(F.col("prv.reportingDate"), F.col("prd.reportingDate")).alias("reportingDate"),
        F.coalesce(F.col("prv.metricID"), F.col("prd.metricID")).alias("metricID"),
        F.coalesce(F.col("prv.providerCode"), F.col("prd.providerCode")).alias("providerCode"),
        F.coalesce(F.col("prd.prod_value"), F.lit(0.0)).alias("prod_value"),
        F.coalesce(F.col("prv.preview_value"), F.lit(0.0)).alias("preview_value")
    )
    .withColumn("difference_value", F.col("preview_value") - F.col("prod_value"))
)

# ----------------------------
# E. Show full comparison table
# ----------------------------
display(
    comparison_df.orderBy("metricID", "providerCode", "reportingDate")
)

# ----------------------------
# F. Keep only positive differences
# ----------------------------
positive_differences_df = comparison_df.filter(F.col("difference_value") > 0)

display(
    positive_differences_df.orderBy("metricID", "providerCode", "reportingDate")
)

# ----------------------------
# G. Row counts
# ----------------------------
counts_df = spark.createDataFrame(
    [
        ("prod_df", prod_df.count()),
        ("prod_agg", prod_agg.count()),
        ("preview_agg", preview_agg.count()),
        ("comparison_df", comparison_df.count()),
        ("positive_differences_df", positive_differences_df.count())
    ],
    ["dataset", "row_count"]
)

display(counts_df)